* 예외 설계

In [ ]:
class AgentError(Exception):
    status_code=400
    code='agent_error'

    def __init__(self, message:str, *, detail:str|None=None):
        super().__init__(message)
        self.message=message
        self.detail=detail

자식 클래스로 세분화

In [ ]:
class AgentError(Exception):
    status_code = 400
    code = "agent_error"  

    def __init__(self, message: str, *, detail: str | None = None):
        super().__init__(message)  # print(e) -> 메세지 나옴 
        self.message = message 
        self.detail = detail 
 
# 요청한 자원이 없다
class NotFound(AgentError):
    status_code=404
    code='not found'

# 자원은 있으나 이 사용자가 접근할 수 없다
class PermissionDenied(AgentError):
    status_code=403
    code='permission_denied'

# 입력값이 규칙에 맞지 않다
class ValidationFailed(AgentError):
    status_code=422
    code='validation_failed'

# 외부 서비스 호출 실패
class ExternalServiceError(AgentError):
    status_code=502
    code='external_service_error'

e=NotFound('문서를 찾을 수 없습니다.', detail='doc_id에 id가 없음')

In [ ]:
def handle(exc):
    if isinstance(exc, AgentError):
        # 의도한 상황
        return {'status':exc.status_code, 'code':exc.code, 'message':exc.message}
    # 예상치 못한 상황
    return {'status':500, 'code':'internal_error', 'message':'서버 오류 발생'}

for exc in [
    NotFound("문서 못찾음"), 
    PermissionDenied("이 문서를 볼 권한이 없습니다."),
    ExternalServiceError("문서 변환 서비스에 연결하지 못하였습니다."),
    KeyError("doc_id")
]:
    print(handle(exc))

In [ ]:
import json

def parse_bad(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        raise ValidationFailed('데이터 형식이 올바르지 않음',detail=str(e))

try:
    parse_bad('{이건 json 아님}')
except ValidationFailed as e:
    print('예외 메세지:',e)
    print('원인:',e.__cause__,)